In [1]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta

Datenbank setup

In [2]:
db_path = 'krankenkasse_poc.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Tabellen erstellen (bleiben bei Neustart erhalten)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS sap_core_versicherte (
        versicherten_id INTEGER PRIMARY KEY,
        geburtsjahr INTEGER
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS sap_shadow_historie (
        historien_id INTEGER PRIMARY KEY AUTOINCREMENT,
        versicherten_id INTEGER,
        tarif_typ TEXT,
        bundesland TEXT,
        valid_from DATE,
        valid_to DATE
    )
''')
conn.commit()

Datenfetcher für historischen Stand

In [3]:
def fetch_historical_state(as_of_date: str) -> pd.DataFrame:
    """
    Holt den exakten Datenstand zu einem bestimmten Datum (für ML-Training).
    """
    query = f"""
        SELECT 
            c.versicherten_id,
            c.geburtsjahr,
            s.tarif_typ,
            s.bundesland
        FROM sap_core_versicherte c
        JOIN sap_shadow_historie s 
          ON c.versicherten_id = s.versicherten_id
        WHERE '{as_of_date}' BETWEEN s.valid_from AND s.valid_to
    """
    return pd.read_sql_query(query, conn)

Update Funktiom

In [ ]:
def update_versicherten_daten(v_id: int, neuer_tarif: str, neues_bundesland: str, gueltig_ab_str: str):
    """
    Pflegt einen neuen Datensatz ein und historisiert den alten (SCD2).
    """
    gueltig_ab_date = datetime.strptime(gueltig_ab_str, '%Y-%m-%d')
    altes_valid_to_str = (gueltig_ab_date - timedelta(days=1)).strftime('%Y-%m-%d')
    
    # Alten Datensatz schließen
    cursor.execute('''
        UPDATE sap_shadow_historie 
        SET valid_to = ? 
        WHERE versicherten_id = ? AND valid_to = '9999-12-31'
    ''', (altes_valid_to_str, v_id))
    
    # Neuen Datensatz anlegen
    cursor.execute('''
        INSERT INTO sap_shadow_historie 
        (versicherten_id, tarif_typ, bundesland, valid_from, valid_to) 
        VALUES (?, ?, ?, ?, '9999-12-31')
    ''', (v_id, neuer_tarif, neues_bundesland, gueltig_ab_str))
    
    conn.commit()
    print(f"Update erfolgreich: ID {v_id} aktualisiert ab {gueltig_ab_str}.")

update_versicherten_daten(
    v_id=1001, 
    neuer_tarif='Premium', 
    neues_bundesland='Berlin', 
    gueltig_ab_str='2026-06-01'
)


Update erfolgreich: ID 1001 aktualisiert ab 2026-06-01.


In [6]:
def update_versicherten_daten(v_id: int, neuer_tarif: str, neues_bundesland: str, gueltig_ab_str: str):
    """
    Pflegt einen neuen Datensatz ein und historisiert den alten (SCD2).
    """
    gueltig_ab_date = datetime.strptime(gueltig_ab_str, '%Y-%m-%d')
    altes_valid_to_str = (gueltig_ab_date - timedelta(days=1)).strftime('%Y-%m-%d')
    
    # Alten Datensatz schließen
    cursor.execute('''
        UPDATE sap_shadow_historie 
        SET valid_to = ? 
        WHERE versicherten_id = ? AND valid_to = '9999-12-31'
    ''', (altes_valid_to_str, v_id))
    
    # Neuen Datensatz anlegen
    cursor.execute('''
        INSERT INTO sap_shadow_historie 
        (versicherten_id, tarif_typ, bundesland, valid_from, valid_to) 
        VALUES (?, ?, ?, ?, '9999-12-31')
    ''', (v_id, neuer_tarif, neues_bundesland, gueltig_ab_str))
    
    conn.commit()
    print(f"Update erfolgreich: ID {v_id} aktualisiert ab {gueltig_ab_str}.")

# Der Aufruf muss auf dieser Ebene stehen, ohne Einrückung:
update_versicherten_daten(
    v_id=1001, 
    neuer_tarif='Premium', 
    neues_bundesland='Berlin', 
    gueltig_ab_str='2026-06-01'
)

Update erfolgreich: ID 1001 aktualisiert ab 2026-06-01.


In [7]:
def export_to_csv():
    """
    Exportiert die Tabellen als CSV für andere Anwendungen.
    """
    pd.read_sql_query("SELECT * FROM sap_core_versicherte", conn).to_csv('export_core.csv', index=False)
    pd.read_sql_query("SELECT * FROM sap_shadow_historie", conn).to_csv('export_shadow.csv', index=False)
    print("CSV-Export abgeschlossen.")

Initiale Daten

In [8]:
cursor.execute("SELECT COUNT(*) FROM sap_core_versicherte")
if cursor.fetchone()[0] == 0:
    cursor.execute('INSERT INTO sap_core_versicherte VALUES (1001, 1985)')
    cursor.execute('''
        INSERT INTO sap_shadow_historie (versicherten_id, tarif_typ, bundesland, valid_from, valid_to) 
        VALUES (1001, 'Basis', 'NRW', '2020-01-01', '9999-12-31')
    ''')
    conn.commit()
    print("Initialer Datensatz wurde angelegt.\n")

Test-Ablauf 

In [9]:
print("--- 1. Aktueller Stand (vor Update) ---")
print(fetch_historical_state('2026-05-21').to_string(index=False))
print("\n")

# Wir tun so, als würde heute ein Tarifwechsel stattfinden
update_versicherten_daten(v_id=1001, neuer_tarif='Premium', neues_bundesland='NRW', gueltig_ab_str='2026-05-21')

print("\n--- 2. Stand nach dem Update ---")
print(fetch_historical_state('2026-05-21').to_string(index=False))
print("\n")

print("--- 3. Wir reisen in die Vergangenheit (Point-in-Time für ML) ---")
print("Was war der Stand im Jahr 2022?")
print(fetch_historical_state('2022-06-15').to_string(index=False))

print("\n")

--- 1. Aktueller Stand (vor Update) ---
 versicherten_id  geburtsjahr tarif_typ bundesland
            1003         1992     Basis        NRW
            1001         1985   Premium        NRW


Update erfolgreich: ID 1001 aktualisiert ab 2026-05-21.

--- 2. Stand nach dem Update ---
 versicherten_id  geburtsjahr tarif_typ bundesland
            1003         1992     Basis        NRW
            1001         1985   Premium        NRW
            1001         1985   Premium        NRW


--- 3. Wir reisen in die Vergangenheit (Point-in-Time für ML) ---
Was war der Stand im Jahr 2022?
 versicherten_id  geburtsjahr tarif_typ bundesland
            1001         1985     Basis        NRW




In [10]:
# CSV Export testen
export_to_csv()

# Am Ende des Notebooks immer brav schließen
conn.close()

CSV-Export abgeschlossen.
